# Example: BBBC014 — Dose-Response Morphological Profiling

Morphological profiling of distinct cell lines (A549 and MCF7) treated with various compounds across multiple concentrations to study dose-response effects.

**Dataset:** [BBBC014](https://bbbc.broadinstitute.org/BBBC014) (2-channel compound dose-response)

**Key Steps:**
1. **Metadata Prep:** Process annotations and calculate log concentrations for dose responses.
2. **Multi-Channel Processing:** Process full field-of-view 2-channel images.
3. **Dose-Response Analysis:** Extract embeddings and compute properties to differentiate cell line responses and visualize dose-dependent morphological shifts.

In [ ]:
import torch
from phenome import PhenoMe, load_dinov2_model
import pandas as pd

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize PhenoMe and load model
pheno = PhenoMe(device=device, seed=42)
wrapper = load_dinov2_model("dinov2_vitg14_reg", device=device)

## 1. Paths & Configuration

Define the paths to the BBBC014 image dataset and the metadata CSV containing compound and concentration information.

In [ ]:
DATA_DIR = "/datasets/BBBC/BBBC014/BBBC014_v1_images"
METADATA_PATH = "/datasets/BBBC/BBBC014/metadata.csv"

## 2. Load Metadata & Preprocessing

We load the metadata and map the cell types to their respective line names (MCF7 and A549). We also calculate log concentrations to facilitate dose-response visualization.

In [3]:
df_metadata = pd.read_csv(METADATA_PATH)

# Map cell types to line names
df_metadata["cell_line"] = df_metadata["cell_type"].apply(
    lambda x: "MCF7" if x in ["A", "B", "C", "D"] else "A549"
)

# Map concentration points to log concentrations
map_concentration = {
    1: -7.0, 2: -7.523, 3: -8.0, 4: -8.523, 5: -9.0, 6: -9.523, 
    7: -10.0, 8: -10.523, 9: -11.0, 10: -11.523, 11: -12.0, 12: -13.0
}
df_metadata["log_concentration"] = df_metadata["concentration_point"].map(map_concentration)

df_metadata.head()

,image_idx,cell_type,concentration_point,suffix,filename_channel_1,filename_channel_2,cell_line,log_concentration
0,1,A,1,0,Channel 1-01-A-01-00,Channel 2-01-A-01-00,MCF7,-7.000
1,2,A,2,0,Channel 1-02-A-02-00,Channel 2-02-A-02-00,MCF7,-7.523
2,3,A,3,0,Channel 1-03-A-03-00,Channel 2-03-A-03-00,MCF7,-8.000
3,4,A,4,0,Channel 1-04-A-04-00,Channel 2-04-A-04-00,MCF7,-8.523
4,5,A,5,0,Channel 1-05-A-05-00,Channel 2-05-A-05-00,MCF7,-9.000


## 3. Multi-Channel Data Discovery

Since BBBC014 uses two channels per image, we specify both filename columns. PhenoMe will automatically load and merge these channels for processing.

In [4]:
from phenome.utils import make_dataframe_metadata_fn

file_df = pheno.find_files(
    str(DATA_DIR),
    metadata_fn=make_dataframe_metadata_fn(
        df_metadata, 
        filename_column=["filename_channel_1", "filename_channel_2"]
    ),
)
file_df.head()

Found 194 files total.
2 files have missing/failed metadata (drop).
Grouped 192 channel files into 96 samples (multi-channel mode).
DataFrame: 96 files, 11 columns.


,file_path,image_idx,cell_type,concentration_point,suffix,filename_channel_1,filename_channel_2,cell_line,log_concentration,id,mask_path
0,[/data3/agonzalez/datasets/BBBC/BBBC014/BBBC01...,1,A,1,0,Channel 1-01-A-01-00,Channel 2-01-A-01-00,MCF7,-7.000,BBBC014_v1_images_Channel_1-01-A-01-00,None
1,[/data3/agonzalez/datasets/BBBC/BBBC014/BBBC01...,2,A,2,0,Channel 1-02-A-02-00,Channel 2-02-A-02-00,MCF7,-7.523,BBBC014_v1_images_Channel_1-02-A-02-00,None
2,[/data3/agonzalez/datasets/BBBC/BBBC014/BBBC01...,3,A,3,0,Channel 1-03-A-03-00,Channel 2-03-A-03-00,MCF7,-8.000,BBBC014_v1_images_Channel_1-03-A-03-00,None
3,[/data3/agonzalez/datasets/BBBC/BBBC014/BBBC01...,4,A,4,0,Channel 1-04-A-04-00,Channel 2-04-A-04-00,MCF7,-8.523,BBBC014_v1_images_Channel_1-04-A-04-00,None
4,[/data3/agonzalez/datasets/BBBC/BBBC014/BBBC01...,5,A,5,0,Channel 1-05-A-05-00,Channel 2-05-A-05-00,MCF7,-9.000,BBBC014_v1_images_Channel_1-05-A-05-00,None


## 4. Feature Extraction

We extract DINOv2 embeddings and compute a full set of morphological properties. Note the use of large `resize_size` and `pad_size` to accommodate full field-of-view images.

In [ ]:
pheno.process_images(
    wrapper,
    batch_size=8,
    channel_mode="split",
    resize_size=1358,
    pad_size=1360,
)

df_properties = pheno.compute_properties(
    property_preset="complete",
    n_jobs=4,
)

## 5. Visualization

Visualize the morphological differences between cell lines and observe dose-dependent shifts using t-SNE and UMAP.

In [ ]:
pheno.plot_tsne(
    color_by="cell_line",
    n_components=2,
)

In [8]:
pheno.plot_umap(
    color_by="log_concentration",
    filters={"cell_line": "A549"},
    n_components=2,
)

## 6. Interactive Exploration

Launch the interactive explorer to inspect the multi-channel images and their relative positions in the phenospace, highlighting how treatments affect cell morphology.

In [9]:
e = pheno.create_interactive_explorer()